# Dev notebook

Connect to the local Postgres DB and explore the CMS SYNPUF sample data with raw SQL.

Prereqs: `just db-up` and `just load-fresh` (or see README).

In [ ]:
import polars as pl
from IPython.display import HTML, display

from phi_guardrails.config import load_config
from phi_guardrails.db import connect

conn = connect(load_config())


def run(sql: str, params: tuple | None = None):
    """Run raw SQL and display the result as an HTML table."""
    with conn.cursor() as cur:
        cur.execute(sql, params or ())
        cols = [d.name for d in cur.description] if cur.description else []
        rows = cur.fetchall()
    display(HTML(f"<p><code>{sql.strip()}</code></p>"))
    return rows, cols


def show(rows, cols, max_rows: int = 10):
    df = pl.DataFrame(rows, schema=cols)
    display(df.head(max_rows))
    print(f"{len(df)} rows")
    return df

## Row counts

In [ ]:
rows, cols = run(
    """
    SELECT 'beneficiary' AS table, COUNT(*) AS n FROM beneficiary
    UNION ALL
    SELECT 'inpatient_claims', COUNT(*) FROM inpatient_claims
    """
)
show(rows, cols)

## Beneficiaries — sample

In [ ]:
rows, cols = run(
    """
    SELECT desynpuf_id, bene_birth_dt, bene_sex_ident_cd, sp_state_code,
           sp_diabetes, sp_cncr, medreimb_ip
    FROM beneficiary
    LIMIT 10
    """
)
show(rows, cols)

## Inpatient claims — sample

In [ ]:
rows, cols = run(
    """
    SELECT desynpuf_id, clm_id, clm_from_dt, clm_thru_dt,
           clm_pmt_amt, icd9_dgns_cd_1, icd9_dgns_cd_2, clm_drg_cd
    FROM inpatient_claims
    LIMIT 10
    """
)
show(rows, cols)

## Join — top spenders

Total inpatient payment per beneficiary, joined to demographics.

In [ ]:
rows, cols = run(
    """
    SELECT b.desynpuf_id,
           b.bene_birth_dt,
           b.bene_sex_ident_cd,
           COUNT(c.clm_id) AS n_claims,
           SUM(c.clm_pmt_amt) AS total_pmt
    FROM beneficiary b
    JOIN inpatient_claims c ON c.desynpuf_id = b.desynpuf_id
    GROUP BY b.desynpuf_id, b.bene_birth_dt, b.bene_sex_ident_cd
    ORDER BY total_pmt DESC
    LIMIT 10
    """
)
show(rows, cols)

## Guardrail demo — query as the agent role

Toggle `DB_USER`/`DB_PASSWORD` in `.env` to `agent001` / `fake-agent-password`,
restart the kernel, and re-run the connection cell. Queries that violate the
agent's grants will fail here.

In [ ]:
rows, cols = run("SELECT current_user, has_table_privilege(current_user, 'beneficiary', 'SELECT')")
show(rows, cols)